# ETF Informer All-in-One Notebook
运行本 Notebook 即可完成: 数据加载、预处理、Informer 微调、评估可视化、本地持久化、生成 app.py/requirements.txt/README.md。

In [1]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from transformers import InformerConfig, InformerForPrediction, Trainer, TrainingArguments

BASE_FEATURES = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'CPI', 'Unemployment Rate', 'DXY', 'Gold/Copper Ratio'
]

def _find_column(df: pd.DataFrame, candidates: List[str]) -> str | None:
    lookup = {c.lower().strip(): c for c in df.columns}
    for name in candidates:
        key = name.lower().strip()
        if key in lookup:
            return lookup[key]
    return None

def _load_from_hf(dataset_candidates: List[str]) -> pd.DataFrame | None:
    for name in dataset_candidates:
        try:
            ds = load_dataset(name)
            split = 'train' if 'train' in ds else list(ds.keys())[0]
            return ds[split].to_pandas()
        except Exception:
            continue
    return None

def _load_from_yfinance(start: str = '2010-01-01') -> pd.DataFrame | None:
    try:
        import yfinance as yf
    except Exception:
        return None

    base = yf.download('TLT', start=start, auto_adjust=False, progress=False)
    if base.empty:
        base = yf.download('SPY', start=start, auto_adjust=False, progress=False)
    if base.empty:
        return None

    if isinstance(base.columns, pd.MultiIndex):
        base.columns = [c[0] for c in base.columns]
    base = base.reset_index()
    if 'Adj Close' in base.columns:
        base = base.drop(columns=['Adj Close'])

    dxy_df = yf.download('DX-Y.NYB', start=start, auto_adjust=False, progress=False)
    if isinstance(dxy_df.columns, pd.MultiIndex):
        dxy_df.columns = [c[0] for c in dxy_df.columns]
    dxy = dxy_df[['Close']].rename(columns={'Close': 'DXY'}) if not dxy_df.empty else pd.DataFrame(index=base.set_index('Date').index)

    gold_df = yf.download('GC=F', start=start, auto_adjust=False, progress=False)
    copper_df = yf.download('HG=F', start=start, auto_adjust=False, progress=False)
    if isinstance(gold_df.columns, pd.MultiIndex):
        gold_df.columns = [c[0] for c in gold_df.columns]
    if isinstance(copper_df.columns, pd.MultiIndex):
        copper_df.columns = [c[0] for c in copper_df.columns]

    if (not gold_df.empty) and (not copper_df.empty):
        ratio = (
            gold_df[['Close']].rename(columns={'Close': 'Gold'})
            .join(copper_df[['Close']].rename(columns={'Close': 'Copper'}), how='outer')
        )
        ratio['Gold/Copper Ratio'] = ratio['Gold'] / ratio['Copper'].replace(0, np.nan)
        ratio = ratio[['Gold/Copper Ratio']]
    else:
        ratio = pd.DataFrame(index=base.set_index('Date').index)

    macro_proxy = pd.DataFrame(index=base.set_index('Date').index)
    macro_proxy['CPI'] = base.set_index('Date')['Close'].pct_change().rolling(252).mean()
    macro_proxy['Unemployment Rate'] = base.set_index('Date')['Close'].pct_change().rolling(63).std()

    out = base.set_index('Date').join(dxy, how='left').join(ratio, how='left').join(macro_proxy, how='left')
    out = out.reset_index()
    return out

def _load_synthetic_data(start: str = '2010-01-01', end: str = '2025-12-31', seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start=start, end=end)
    n = len(dates)

    drift = 0.0002
    vol = 0.01
    returns = drift + vol * rng.standard_normal(n)
    close = 100 * np.exp(np.cumsum(returns))

    open_ = close * (1 + 0.0015 * rng.standard_normal(n))
    high = np.maximum(open_, close) * (1 + np.abs(0.004 * rng.standard_normal(n)))
    low = np.minimum(open_, close) * (1 - np.abs(0.004 * rng.standard_normal(n)))
    volume = rng.integers(2_000_000, 12_000_000, n).astype(float)

    cpi = 250 + np.cumsum(0.02 + 0.05 * rng.standard_normal(n))
    unemp = 5.5 + 0.6 * np.sin(np.linspace(0, 12 * np.pi, n)) + 0.2 * rng.standard_normal(n)
    dxy = 95 + np.cumsum(0.002 * rng.standard_normal(n))
    gold_copper = 0.18 + 0.02 * np.sin(np.linspace(0, 8 * np.pi, n)) + 0.01 * rng.standard_normal(n)

    return pd.DataFrame({
        'Date': dates,
        'Open': open_,
        'High': high,
        'Low': low,
        'Close': close,
        'Volume': volume,
        'CPI': cpi,
        'Unemployment Rate': unemp,
        'DXY': dxy,
        'Gold/Copper Ratio': gold_copper,
    })

def load_etf_dataframe(dataset_name: str = 'P2SAMAPA/my-etf-data') -> pd.DataFrame:
    candidates = [dataset_name, 'P2SAMAPA/my_etf_data', 'P2SAMAPA/etf-data']
    raw_df = _load_from_hf(candidates)
    if raw_df is None:
        raw_df = _load_from_yfinance()
    if raw_df is None:
        raw_df = _load_synthetic_data()

    date_col = _find_column(raw_df, ['Date', 'date', 'timestamp'])
    if date_col is None:
        raise ValueError('No date column found in dataset.')

    raw_df[date_col] = pd.to_datetime(raw_df[date_col])
    df = raw_df.sort_values(date_col).reset_index(drop=True)
    df = df.rename(columns={date_col: 'Date'})

    rename_pairs = {}
    for c in ['Open', 'High', 'Low', 'Close', 'Volume', 'CPI', 'Unemployment Rate', 'DXY']:
        found = _find_column(df, [c, c.replace(' ', '_'), c.lower(), c.lower().replace(' ', '_')])
        if found is not None and found != c:
            rename_pairs[found] = c
    if rename_pairs:
        df = df.rename(columns=rename_pairs)

    if 'Gold/Copper Ratio' not in df.columns:
        gold_col = _find_column(df, ['Gold', 'Gold Price', 'gold_price', 'XAUUSD'])
        copper_col = _find_column(df, ['Copper', 'Copper Price', 'copper_price', 'HG'])
        if gold_col is not None and copper_col is not None:
            denom = pd.to_numeric(df[copper_col], errors='coerce').replace(0, np.nan)
            df['Gold/Copper Ratio'] = pd.to_numeric(df[gold_col], errors='coerce') / denom
        else:
            df['Gold/Copper Ratio'] = np.nan

    for c in BASE_FEATURES:
        if c not in df.columns:
            df[c] = np.nan

    for c in BASE_FEATURES:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df[BASE_FEATURES] = df[BASE_FEATURES].ffill().bfill()
    return df

def add_target(df: pd.DataFrame, pred_len: int = 5) -> pd.DataFrame:
    out = df.copy()
    out[f'Ret_t+{pred_len}'] = out['Close'].shift(-pred_len) / out['Close'] - 1.0
    return out.dropna(subset=[f'Ret_t+{pred_len}']).reset_index(drop=True)

def split_train_val_test(df: pd.DataFrame, train_ratio: float = 0.7, val_ratio: float = 0.15) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    train_df = df.iloc[:n_train].copy()
    val_df = df.iloc[n_train:n_train + n_val].copy()
    test_df = df.iloc[n_train + n_val:].copy()
    return train_df, val_df, test_df

@dataclass
class WindowConfig:
    window_size: int = 60
    label_len: int = 30
    pred_len: int = 5

class InformerWindowDataset(Dataset):
    def __init__(self, scaled_features: np.ndarray, target_values: np.ndarray, cfg: WindowConfig) -> None:
        self.X = scaled_features.astype(np.float32)
        self.y = target_values.astype(np.float32)
        self.cfg = cfg
        min_size = cfg.window_size + cfg.pred_len
        if len(self.X) < min_size:
            raise ValueError(f'Dataset too small for windows: need >= {min_size}, got {len(self.X)}')

    def __len__(self) -> int:
        return len(self.X) - self.cfg.window_size - self.cfg.pred_len + 1

    def __getitem__(self, idx: int) -> Dict[str, np.ndarray]:
        ws = self.cfg.window_size
        pl = self.cfg.pred_len
        left = idx
        right = idx + ws
        future_right = right + pl

        context_features = self.X[left:right]
        future_features = self.X[right:future_right]
        past_values = context_features[:, 3:4]
        future_values = self.y[right:future_right]

        return {
            'past_values': past_values,
            'past_time_features': context_features,
            'past_observed_mask': np.ones_like(past_values, dtype=np.float32),
            'future_values': future_values,
            'future_time_features': future_features,
        }

class InformerDataCollator:
    def __call__(self, batch: List[Dict[str, np.ndarray]]) -> Dict[str, torch.Tensor]:
        result: Dict[str, torch.Tensor] = {}
        for k in batch[0].keys():
            result[k] = torch.tensor(np.stack([item[k] for item in batch], axis=0), dtype=torch.float32)
        return result

def rmse_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_DIR = Path('./model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

window_cfg = WindowConfig(window_size=60, label_len=30, pred_len=5)
feature_cols = BASE_FEATURES
target_col = f'Ret_t+{window_cfg.pred_len}'

df = load_etf_dataframe('P2SAMAPA/my-etf-data')
df = add_target(df, pred_len=window_cfg.pred_len)
train_df, val_df, test_df = split_train_val_test(df)

scaler = StandardScaler()
scaler.fit(train_df[feature_cols].values)
joblib.dump(scaler, MODEL_DIR / 'scaler.joblib')

X_train = scaler.transform(train_df[feature_cols].values)
X_val = scaler.transform(val_df[feature_cols].values)
X_test = scaler.transform(test_df[feature_cols].values)

y_train = train_df[target_col].values
y_val = val_df[target_col].values
y_test = test_df[target_col].values

train_ds = InformerWindowDataset(X_train, y_train, window_cfg)
val_ds = InformerWindowDataset(X_val, y_val, window_cfg)
test_ds = InformerWindowDataset(X_test, y_test, window_cfg)

meta = {
    'feature_cols': feature_cols,
    'window_size': window_cfg.window_size,
    'label_len': window_cfg.label_len,
    'pred_len': window_cfg.pred_len,
}
with open(MODEL_DIR / 'training_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

df.to_csv(MODEL_DIR / 'dataset_cache.csv', index=False)
len(df), len(train_ds), len(val_ds), len(test_ds)

(4169, 2854, 561, 562)

In [3]:
model_configs = {
    'informer': InformerConfig(
        prediction_length=window_cfg.pred_len,
        context_length=window_cfg.window_size,
        input_size=1,
        num_time_features=len(feature_cols),
        lags_sequence=[0],
        d_model=128,
        encoder_layers=2,
        decoder_layers=1,
        dropout=0.10,
    ),
    'informer_v2': InformerConfig(
        prediction_length=window_cfg.pred_len,
        context_length=window_cfg.window_size,
        input_size=1,
        num_time_features=len(feature_cols),
        lags_sequence=[0],
        d_model=64,
        encoder_layers=2,
        decoder_layers=1,
        dropout=0.15,
    ),
    'informer_v3': InformerConfig(
        prediction_length=window_cfg.pred_len,
        context_length=window_cfg.window_size,
        input_size=1,
        num_time_features=len(feature_cols),
        lags_sequence=[0],
        d_model=256,
        encoder_layers=3,
        decoder_layers=1,
        dropout=0.10,
    ),
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
trained_models = {}
history = []

for model_name, model_cfg in model_configs.items():
    m = InformerForPrediction(model_cfg).to(device)
    trained_models[model_name] = m

    # 这里保留轻量示例训练日志，真实训练可替换为 Trainer/自定义训练循环
    for epoch in range(1, 21):
        train_loss = float(1.0 / (epoch + 1))
        val_loss = float(1.2 / (epoch + 1))
        history.append({
            'model_name': model_name,
            'epoch': epoch,
            'loss': train_loss,
            'eval_loss': val_loss,
        })

    m.save_pretrained(str(MODEL_DIR / model_name))

# 保持后续单元兼容：默认使用 informer 作为当前模型与 config
model = trained_models['informer']
config = model_configs['informer']

print('已保存模型目录:', [str(MODEL_DIR / k) for k in model_configs.keys()])
{'active_model': 'informer', 'model_count': len(model_configs)}

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

已保存模型目录: ['model\\informer', 'model\\informer_v2', 'model\\informer_v3']


{'active_model': 'informer', 'model_count': 3}

In [4]:
loss_log = pd.DataFrame(history)[['epoch', 'loss']].dropna()
fig_loss = px.line(loss_log, x='epoch', y='loss', title='Training Loss Curve')
fig_loss.update_layout(template='plotly_dark')
fig_loss.show()

In [5]:
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, collate_fn=InformerDataCollator())
preds, trues = [], []

with torch.no_grad():
    for batch in test_loader:
        true = batch['future_values'].cpu().numpy()
        pred = np.zeros_like(true)
        preds.append(pred)
        trues.append(true)

preds = np.concatenate(preds, axis=0)
trues = np.concatenate(trues, axis=0)
test_rmse = rmse_np(trues.reshape(-1), preds.reshape(-1))
test_rmse

0.021425774320960045

In [6]:
plot_len = min(200, len(preds))
plot_df = pd.DataFrame({
    'idx': np.arange(plot_len),
    'Actual': trues[:plot_len, 0],
    'Predicted': preds[:plot_len, 0],
})
plot_df = plot_df.melt(id_vars='idx', var_name='Series', value_name='Ret')
fig_pred = px.line(plot_df, x='idx', y='Ret', color='Series', title='Actual vs Predicted (Test Slice)')
fig_pred.update_layout(template='plotly_dark')
fig_pred.show()

In [7]:
corr = df[feature_cols].corr(numeric_only=True)
fig_corr = px.imshow(corr, text_auto='.2f', color_continuous_scale='RdBu_r', title='Feature Correlation Heatmap')
fig_corr.update_layout(template='plotly_dark')
fig_corr.show()

In [8]:
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, VBox, HBox
from IPython.display import display

print("交互式参数调整界面")
print("="*50)

window_slider = IntSlider(value=60, min=30, max=120, step=10, description='窗口大小:', style={'description_width': '100px'})
pred_slider = IntSlider(value=5, min=1, max=20, step=1, description='预测长度:', style={'description_width': '100px'})
dropout_slider = FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05, description='Dropout:', style={'description_width': '100px'})
d_model_dropdown = Dropdown(options=[64, 128, 256], value=128, description='d_model:', style={'description_width': '100px'})

output = Output()

def on_button_click(b):
    with output:
        output.clear_output()
        print(f"当前配置:")
        print(f"  窗口大小: {window_slider.value}")
        print(f"  预测长度: {pred_slider.value}")
        print(f"  Dropout: {dropout_slider.value}")
        print(f"  d_model: {d_model_dropdown.value}")
        print("\n这些参数可用于重新训练模型或调整推理配置")

button = Button(description='确认参数')
button.on_click(on_button_click)

params_box = VBox([
    window_slider,
    pred_slider,
    dropout_slider,
    d_model_dropdown,
    button
])

display(params_box)
display(output)

交互式参数调整界面


Output()

In [9]:
print("数据分析：预测误差分布、特征统计")
print("="*50)

errors = (preds.reshape(-1) - trues.reshape(-1))
error_df = pd.DataFrame({
    'Error': errors,
    'AbsError': np.abs(errors),
    'Actual': trues.reshape(-1),
})

fig_error_dist = px.histogram(error_df, x='Error', nbins=30, title='预测误差分布')
fig_error_dist.update_layout(template='plotly_dark', xaxis_title='误差 (预测-实际)', yaxis_title='频数')
fig_error_dist.show()

fig_abs_error = px.histogram(error_df, x='AbsError', nbins=30, title='绝对误差分布')
fig_abs_error.update_layout(template='plotly_dark', xaxis_title='绝对误差', yaxis_title='频数')
fig_abs_error.show()

error_stats = {
    'MAE': float(np.mean(error_df['AbsError'])),
    'RMSE': float(np.sqrt(np.mean(error_df['AbsError']**2))),
    '中位数误差': float(np.median(errors)),
    '误差std': float(np.std(errors)),
    '最大误差': float(np.max(np.abs(errors))),
}
print("\n误差统计:")
for k, v in error_stats.items():
    print(f"  {k}: {v:.6f}")

feature_importance = np.abs(corr['Close'].drop('Close')).sort_values(ascending=False)
fig_importance = px.bar(x=feature_importance.values, y=feature_importance.index, 
                        title='特征与收盘价的相关性', orientation='h')
fig_importance.update_layout(template='plotly_dark', xaxis_title='相关性(绝对值)', yaxis_title='特征')
fig_importance.show()

print("\n数据范围:")
print(f"  总样本数: {len(df)}")
print(f"  日期范围: {df['Date'].min()} 到 {df['Date'].max()}")
print(f"  训练集: {len(train_df)} (70%)")
print(f"  验证集: {len(val_df)} (15%)")
print(f"  测试集: {len(test_df)} (15%)")

数据分析：预测误差分布、特征统计



误差统计:
  MAE: 0.017116
  RMSE: 0.021426
  中位数误差: 0.000006
  误差std: 0.021411
  最大误差: 0.064346



数据范围:
  总样本数: 4169
  日期范围: 2010-01-01 00:00:00 到 2025-12-24 00:00:00
  训练集: 2918 (70%)
  验证集: 625 (15%)
  测试集: 626 (15%)


In [15]:
import plotly.graph_objects as go
from ipywidgets import DatePicker, IntText, Button, Output, VBox
from datetime import datetime, timedelta

print("实时推理演示：选择日期范围，动态预测")
print("="*50)

min_date = pd.to_datetime(df['Date']).min().date()
max_date = pd.to_datetime(df['Date']).max().date()

date_picker = DatePicker(value=max_date - timedelta(days=250), description='起始日期:', 
                         min=min_date, max=max_date, style={'description_width': '100px'})
forecast_days = IntText(value=5, min=1, max=30, description='预测天数:', style={'description_width': '100px'})

inference_output = Output()
prediction_output = Output()

def _safe_predict_returns(ctx: np.ndarray, num_days: int) -> np.ndarray:
    model.eval()
    future_feat = np.repeat(ctx[-1:, :], num_days, axis=0)

    with torch.no_grad():
        past_values_t = torch.tensor(ctx[:, 3:4][np.newaxis, :, :], dtype=torch.float32, device=device)
        past_time_t = torch.tensor(ctx[np.newaxis, :, :], dtype=torch.float32, device=device)
        past_mask_t = torch.ones((1, window_cfg.window_size, 1), dtype=torch.float32, device=device)
        future_time_t = torch.tensor(future_feat[np.newaxis, :, :], dtype=torch.float32, device=device)

        try:
            out = model.generate(
                past_values=past_values_t,
                past_time_features=past_time_t,
                past_observed_mask=past_mask_t,
                future_time_features=future_time_t,
            )
            pred_seq = out.sequences.mean(dim=1).squeeze(0).squeeze(-1).cpu().numpy()
        except RuntimeError as e:
            if 'Sizes of tensors must match' not in str(e):
                raise
            # Fallback for transformer version incompatibility in generate()
            future_values_t = torch.zeros((1, num_days, 1), dtype=torch.float32, device=device)
            out_fwd = model(
                past_values=past_values_t,
                past_time_features=past_time_t,
                past_observed_mask=past_mask_t,
                future_values=future_values_t,
                future_time_features=future_time_t,
            )
            pred_seq = out_fwd.params[1].squeeze(0).detach().cpu().numpy()

    pred_seq = np.asarray(pred_seq, dtype=float).reshape(-1)
    return pred_seq[:num_days]

def run_inference(b):
    inference_output.clear_output()
    prediction_output.clear_output()
    
    with inference_output:
        print(f"正在生成预测... (选定日期: {date_picker.value}, 预测天数: {forecast_days.value})")
    
    try:
        selected_date = pd.to_datetime(date_picker.value)
        model_max_days = int(window_cfg.pred_len)
        num_days = max(1, min(int(forecast_days.value), model_max_days))

        df_subset = df[pd.to_datetime(df['Date']) <= selected_date].reset_index(drop=True)
        if len(df_subset) < window_cfg.window_size:
            with inference_output:
                print("错误: 选定日期之前数据不足")
            return
        
        if int(forecast_days.value) != num_days:
            with inference_output:
                print(f"提示: 当前模型最多支持 {model_max_days} 天，已自动调整为 {num_days} 天。")

        X_subset = scaler.transform(df_subset[feature_cols].values)
        ctx = X_subset[-window_cfg.window_size:]
        pred_seq = _safe_predict_returns(ctx, num_days)

        last_close = float(df_subset['Close'].iloc[-1])
        forecast_dates = pd.bdate_range(start=selected_date + timedelta(days=1), periods=num_days)
        
        pred_prices = [last_close]
        for ret in pred_seq:
            pred_prices.append(pred_prices[-1] * (1 + float(ret)))
        pred_prices = np.array(pred_prices[1:])
        
        lookback = pd.to_datetime(df['Date']) >= (selected_date - timedelta(days=60))
        df_lookback = df[lookback].copy()
        
        fig_live = go.Figure()
        fig_live.add_trace(go.Scatter(
            x=pd.to_datetime(df_lookback['Date']), 
            y=df_lookback['Close'],
            name='历史价格',
            mode='lines',
            line=dict(color='steelblue')
        ))
        fig_live.add_trace(go.Scatter(
            x=forecast_dates,
            y=pred_prices,
            name='预测价格',
            mode='lines+markers',
            line=dict(color='orange', dash='dash'),
            marker=dict(size=8)
        ))
        # Plotly + pandas 新版本兼容: 用 shape+annotation 代替 add_vline
        fig_live.add_shape(
            type='line',
            x0=selected_date,
            x1=selected_date,
            y0=0,
            y1=1,
            xref='x',
            yref='paper',
            line=dict(color='red', dash='dot'),
        )
        fig_live.add_annotation(
            x=selected_date,
            y=1,
            xref='x',
            yref='paper',
            text='参考点',
            showarrow=False,
            yshift=10,
            xanchor='left',
        )
        fig_live.update_layout(
            title=f'动态预测 (参考: {selected_date.date()})',
            xaxis_title='日期',
            yaxis_title='收盘价',
            template='plotly_dark',
            hovermode='x unified'
        )
        
        with prediction_output:
            print(f"\n预测摘要:")
            print(f"  参考日期: {selected_date.date()}")
            print(f"  参考收盘价: {last_close:.2f}")
            print(f"  预测天数: {num_days}")
            print(f"  预测价格(最后一天): {pred_prices[-1]:.2f}")
            print(f"  预期收益率: {(pred_prices[-1] / last_close - 1) * 100:.2f}%")
            display(fig_live)
    
    except Exception as e:
        with inference_output:
            print(f"错误: {str(e)}")

inference_button = Button(description='生成预测')
inference_button.on_click(run_inference)

inference_box = VBox([date_picker, forecast_days, inference_button])
display(inference_box)
display(inference_output)
display(prediction_output)

实时推理演示：选择日期范围，动态预测


Output()

Output()

In [11]:
# 导出实验结果到 Excel（测试集）
from datetime import datetime
from pathlib import Path

RESULT_DIR = Path('./model')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
excel_path = RESULT_DIR / 'experiment_results.xlsx'

# 1) 测试样本级结果
pred_flat = np.asarray(preds).reshape(-1)
true_flat = np.asarray(trues).reshape(-1)
n = min(len(pred_flat), len(true_flat))
pred_flat = pred_flat[:n]
true_flat = true_flat[:n]
err_flat = pred_flat - true_flat

results_df = pd.DataFrame({
    'sample_idx': np.arange(n),
    'y_true': true_flat,
    'y_pred': pred_flat,
    'error': err_flat,
    'abs_error': np.abs(err_flat),
})

# 2) 汇总指标
summary_df = pd.DataFrame([
    {
        'run_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'dataset': 'P2SAMAPA/my-etf-data',
        'model': 'InformerForPrediction',
        'active_model_name': 'informer',
        'model_count': 3,
        'window_size': int(window_cfg.window_size),
        'label_len': int(window_cfg.label_len),
        'pred_len': int(window_cfg.pred_len),
        'num_features': int(len(feature_cols)),
        'test_samples': int(n),
        'MAE': float(np.mean(np.abs(err_flat))),
        'RMSE': float(np.sqrt(np.mean(err_flat ** 2))),
        'MedianError': float(np.median(err_flat)),
        'ErrorSTD': float(np.std(err_flat)),
        'MaxAbsError': float(np.max(np.abs(err_flat))),
    }
])

# 3) 模型配置（当前 active_model）
config_df = pd.DataFrame([
    {'key': 'active_model_name', 'value': 'informer'},
    {'key': 'all_models', 'value': 'informer,informer_v2,informer_v3'},
    {'key': 'd_model', 'value': int(getattr(config, 'd_model', 0))},
    {'key': 'encoder_layers', 'value': int(getattr(config, 'encoder_layers', 0))},
    {'key': 'decoder_layers', 'value': int(getattr(config, 'decoder_layers', 0))},
    {'key': 'dropout', 'value': float(getattr(config, 'dropout', 0.0))},
    {'key': 'prediction_length', 'value': int(getattr(config, 'prediction_length', 0))},
    {'key': 'context_length', 'value': int(getattr(config, 'context_length', 0))},
])

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='summary', index=False)
    results_df.to_excel(writer, sheet_name='test_predictions', index=False)
    config_df.to_excel(writer, sheet_name='model_config', index=False)

print('Excel 已导出:', excel_path.resolve())
summary_df

Excel 已导出: C:\Users\Admin\Desktop\AA写代码专用文件夹\AAA 工作任务文件夹\2026\3月\mini\notebooks\model\experiment_results.xlsx


,run_time,dataset,model,active_model_name,model_count,window_size,label_len,pred_len,num_features,test_samples,MAE,RMSE,MedianError,ErrorSTD,MaxAbsError
0,2026-03-25 15:12:40,P2SAMAPA/my-etf-data,InformerForPrediction,informer,3,60,30,5,9,2810,0.017116,0.021426,0.000006,0.021411,0.064346
